In [1]:
import os

print("Available input datasets:\n")

for item in os.listdir("/kaggle/input"):
    print(item)

Available input datasets:

datasets


In [2]:
import os

base = "/kaggle/input/datasets/smnahian/low-light-final/vehicle_lowlight_final"

original_images = os.path.join(
    base,
    "vehicle_dataset",
    "Images"
)

annotations = os.path.join(
    base,
    "vehicle_dataset",
    "Annotations"
)

enhanced_images = os.path.join(
    base,
    "enhanced_vehicle_dataset",
    "Images"
)

print("Base exists:", os.path.exists(base))
print("Original images exists:", os.path.exists(original_images))
print("Annotations exists:", os.path.exists(annotations))
print("Enhanced images exists:", os.path.exists(enhanced_images))

Base exists: True
Original images exists: True
Annotations exists: True
Enhanced images exists: True


In [3]:
import os
from collections import Counter

base = "/kaggle/input/datasets/smnahian/low-light-final/vehicle_lowlight_final"

original_root = os.path.join(
    base,
    "vehicle_dataset",
    "Images"
)

annotation_root = os.path.join(
    base,
    "vehicle_dataset",
    "Annotations"
)

enhanced_root = os.path.join(
    base,
    "enhanced_vehicle_dataset",
    "Images"
)

valid_ext = (
    ".jpg", ".jpeg", ".png",
    ".JPG", ".JPEG", ".PNG"
)

def count_images(root):
    counts = Counter()
    total = 0

    for class_name in sorted(os.listdir(root)):
        class_dir = os.path.join(root, class_name)

        if not os.path.isdir(class_dir):
            continue

        count = sum(
            1 for f in os.listdir(class_dir)
            if f.endswith(valid_ext)
        )

        counts[class_name] = count
        total += count

    return total, counts


def count_annotations(root):
    counts = Counter()
    total = 0

    for class_name in sorted(os.listdir(root)):
        class_dir = os.path.join(root, class_name)

        if not os.path.isdir(class_dir):
            continue

        count = sum(
            1 for f in os.listdir(class_dir)
            if f.lower().endswith(".txt")
        )

        counts[class_name] = count
        total += count

    return total, counts


original_total, original_counts = count_images(original_root)
enhanced_total, enhanced_counts = count_images(enhanced_root)
annotation_total, annotation_counts = count_annotations(annotation_root)

print("=" * 60)
print("FINAL INPUT DATASET COUNT CHECK")
print("=" * 60)

print("\nOriginal images:", original_total)
print("Enhanced images:", enhanced_total)
print("Annotations:", annotation_total)

print("\nClass-wise comparison:\n")

classes = sorted(
    set(original_counts)
    | set(enhanced_counts)
    | set(annotation_counts)
)

for cls in classes:
    print(
        f"{cls:12s} | "
        f"Original: {original_counts[cls]:4d} | "
        f"Enhanced: {enhanced_counts[cls]:4d} | "
        f"Annotations: {annotation_counts[cls]:4d}"
    )

print("\n" + "=" * 60)

if (
    original_total == 2997
    and enhanced_total == 2997
    and annotation_total == 2997
    and original_counts == enhanced_counts == annotation_counts
):
    print("✅ FINAL INPUT DATASET VERIFIED")
else:
    print("❌ COUNT OR CLASS MISMATCH FOUND")

print("=" * 60)

FINAL INPUT DATASET COUNT CHECK

Original images: 2997
Enhanced images: 2997
Annotations: 2997

Class-wise comparison:

Bicycle      | Original:  651 | Enhanced:  651 | Annotations:  651
Boat         | Original:  679 | Enhanced:  679 | Annotations:  679
Bus          | Original:  527 | Enhanced:  527 | Annotations:  527
Car          | Original:  638 | Enhanced:  638 | Annotations:  638
Motorbike    | Original:  502 | Enhanced:  502 | Annotations:  502

✅ FINAL INPUT DATASET VERIFIED


In [4]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

base = "/kaggle/input/datasets/smnahian/low-light-final/vehicle_lowlight_final"

original_root = os.path.join(
    base,
    "vehicle_dataset",
    "Images"
)

classes = ["Bicycle", "Boat", "Bus", "Car", "Motorbike"]

rows = []

for class_name in classes:
    class_dir = os.path.join(original_root, class_name)

    for filename in sorted(os.listdir(class_dir)):
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):
            rows.append({
                "class": class_name,
                "filename": filename,
                "relative_path": os.path.join(class_name, filename)
            })

df = pd.DataFrame(rows)

print("Total clean images:", len(df))

# ============================================================
# 70% TRAIN, 30% TEMP
# ============================================================

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["class"]
)

# ============================================================
# TEMP -> 15% VAL + 15% TEST
# ============================================================

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["class"]
)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

split_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

# Save fixed split
split_path = "/kaggle/working/vehicle_split_70_15_15.csv"

split_df.to_csv(
    split_path,
    index=False
)

print("\nSplit counts:")
print(split_df["split"].value_counts())

print("\nClass x Split:")
print(
    pd.crosstab(
        split_df["class"],
        split_df["split"]
    )
)

print("\nSaved to:")
print(split_path)

Total clean images: 2997

Split counts:
split
train    2097
val       450
test      450
Name: count, dtype: int64

Class x Split:
split      test  train  val
class                      
Bicycle      98    456   97
Boat        102    475  102
Bus          79    369   79
Car          96    446   96
Motorbike    75    351   76

Saved to:
/kaggle/working/vehicle_split_70_15_15.csv


In [5]:
import os
import pandas as pd

# ============================================================
# PATHS
# ============================================================

base = "/kaggle/input/datasets/smnahian/low-light-final/vehicle_lowlight_final"

original_root = os.path.join(
    base,
    "vehicle_dataset",
    "Images"
)

enhanced_root = os.path.join(
    base,
    "enhanced_vehicle_dataset",
    "Images"
)

annotation_root = os.path.join(
    base,
    "vehicle_dataset",
    "Annotations"
)

split_path = "/kaggle/working/vehicle_split_70_15_15.csv"

# ============================================================
# LOAD FIXED SPLIT
# ============================================================

split_df = pd.read_csv(split_path)

missing_original = []
missing_enhanced = []
missing_annotation = []

# ============================================================
# CHECK EVERY SPLIT IMAGE
# ============================================================

for _, row in split_df.iterrows():

    class_name = row["class"]
    filename = row["filename"]

    original_path = os.path.join(
        original_root,
        class_name,
        filename
    )

    enhanced_path = os.path.join(
        enhanced_root,
        class_name,
        filename
    )

    # ExDark annotation naming:
    # image.jpg -> image.jpg.txt
    annotation_path = os.path.join(
        annotation_root,
        class_name,
        filename + ".txt"
    )

    if not os.path.exists(original_path):
        missing_original.append(
            f"{class_name}/{filename}"
        )

    if not os.path.exists(enhanced_path):
        missing_enhanced.append(
            f"{class_name}/{filename}"
        )

    if not os.path.exists(annotation_path):
        missing_annotation.append(
            f"{class_name}/{filename}.txt"
        )

# ============================================================
# SPLIT OVERLAP CHECK
# ============================================================

train_set = set(
    split_df.loc[
        split_df["split"] == "train",
        "relative_path"
    ]
)

val_set = set(
    split_df.loc[
        split_df["split"] == "val",
        "relative_path"
    ]
)

test_set = set(
    split_df.loc[
        split_df["split"] == "test",
        "relative_path"
    ]
)

train_val_overlap = train_set & val_set
train_test_overlap = train_set & test_set
val_test_overlap = val_set & test_set

# ============================================================
# RESULTS
# ============================================================

print("=" * 65)
print("SPLIT + DATASET CONSISTENCY CHECK")
print("=" * 65)

print("\nSplit rows:", len(split_df))

print("\nTrain:", len(train_set))
print("Validation:", len(val_set))
print("Test:", len(test_set))

print("\nMissing original images:", len(missing_original))
print("Missing enhanced images:", len(missing_enhanced))
print("Missing annotations:", len(missing_annotation))

print("\nTrain-Val overlap:", len(train_val_overlap))
print("Train-Test overlap:", len(train_test_overlap))
print("Val-Test overlap:", len(val_test_overlap))

print("\n" + "=" * 65)

if (
    len(split_df) == 2997
    and len(missing_original) == 0
    and len(missing_enhanced) == 0
    and len(missing_annotation) == 0
    and len(train_val_overlap) == 0
    and len(train_test_overlap) == 0
    and len(val_test_overlap) == 0
):
    print("✅ SPLIT AND DATASET CONSISTENCY VERIFIED")
    print("Original + Enhanced + Annotation pairing is complete.")
    print("No train/val/test leakage detected.")
else:
    print("❌ CONSISTENCY ISSUE FOUND")

print("=" * 65)

if missing_original:
    print("\nFirst missing originals:")
    print(missing_original[:10])

if missing_enhanced:
    print("\nFirst missing enhanced:")
    print(missing_enhanced[:10])

if missing_annotation:
    print("\nFirst missing annotations:")
    print(missing_annotation[:10])

SPLIT + DATASET CONSISTENCY CHECK

Split rows: 2997

Train: 2097
Validation: 450
Test: 450

Missing original images: 0
Missing enhanced images: 0
Missing annotations: 0

Train-Val overlap: 0
Train-Test overlap: 0
Val-Test overlap: 0

✅ SPLIT AND DATASET CONSISTENCY VERIFIED
Original + Enhanced + Annotation pairing is complete.
No train/val/test leakage detected.


In [6]:
import os
import cv2

# ============================================================
# PATHS
# ============================================================

base = "/kaggle/input/datasets/smnahian/low-light-final/vehicle_lowlight_final"

image_root = os.path.join(base, "vehicle_dataset", "Images")
annotation_root = os.path.join(base, "vehicle_dataset", "Annotations")

# Fixed class mapping
class_map = {
    "Bicycle": 0,
    "Boat": 1,
    "Bus": 2,
    "Car": 3,
    "Motorbike": 4
}

# ============================================================
# TAKE ONE SAMPLE FROM OUR LOCKED SPLIT
# ============================================================

sample = split_df.iloc[0]

class_name = sample["class"]
filename = sample["filename"]

image_path = os.path.join(
    image_root,
    class_name,
    filename
)

annotation_path = os.path.join(
    annotation_root,
    class_name,
    filename + ".txt"
)

# ============================================================
# READ IMAGE
# ============================================================

img = cv2.imread(image_path)

if img is None:
    raise ValueError("Could not read sample image.")

img_h, img_w = img.shape[:2]

print("=" * 65)
print("YOLO ANNOTATION CONVERSION TEST")
print("=" * 65)

print("\nImage:")
print(f"{class_name}/{filename}")

print(f"\nImage size: {img_w} x {img_h}")

# ============================================================
# READ + CONVERT ANNOTATION
# ============================================================

converted_labels = []

with open(annotation_path, "r") as f:
    lines = f.readlines()

for line in lines:

    line = line.strip()

    # Skip header / empty lines
    if not line or line.startswith("%"):
        continue

    parts = line.split()

    object_class = parts[0]

    # Ignore anything outside our 5 vehicle classes
    if object_class not in class_map:
        continue

    x_left = float(parts[1])
    y_top = float(parts[2])
    box_w = float(parts[3])
    box_h = float(parts[4])

    # --------------------------------------------------------
    # bbGt -> YOLO
    # --------------------------------------------------------

    x_center = (x_left + box_w / 2) / img_w
    y_center = (y_top + box_h / 2) / img_h

    norm_w = box_w / img_w
    norm_h = box_h / img_h

    class_id = class_map[object_class]

    converted_labels.append(
        [
            class_id,
            x_center,
            y_center,
            norm_w,
            norm_h
        ]
    )

    print("\nOriginal bbGt:")
    print(
        f"{object_class} "
        f"x={x_left}, y={y_top}, "
        f"w={box_w}, h={box_h}"
    )

    print("Converted YOLO:")
    print(
        f"{class_id} "
        f"{x_center:.6f} "
        f"{y_center:.6f} "
        f"{norm_w:.6f} "
        f"{norm_h:.6f}"
    )

# ============================================================
# SANITY CHECK
# ============================================================

valid = True

for label in converted_labels:
    _, xc, yc, w, h = label

    if not (
        0 <= xc <= 1 and
        0 <= yc <= 1 and
        0 < w <= 1 and
        0 < h <= 1
    ):
        valid = False

print("\n" + "=" * 65)

print("Vehicle objects found:", len(converted_labels))

if valid and len(converted_labels) > 0:
    print("✅ SAMPLE YOLO CONVERSION PASSED")
    print("All normalized coordinates are within valid range.")
else:
    print("❌ SAMPLE YOLO CONVERSION FAILED")

print("=" * 65)

YOLO ANNOTATION CONVERSION TEST

Image:
Bus/2015_02134.jpg

Image size: 640 x 640

Original bbGt:
Bus x=91.0, y=399.0, w=546.0, h=185.0
Converted YOLO:
2 0.568750 0.767969 0.853125 0.289062

Vehicle objects found: 1
✅ SAMPLE YOLO CONVERSION PASSED
All normalized coordinates are within valid range.


In [7]:
import os
import shutil
import cv2
import pandas as pd

# ============================================================
# PATHS
# ============================================================

base = "/kaggle/input/datasets/smnahian/low-light-final/vehicle_lowlight_final"

original_root = os.path.join(
    base, "vehicle_dataset", "Images"
)

enhanced_root = os.path.join(
    base, "enhanced_vehicle_dataset", "Images"
)

annotation_root = os.path.join(
    base, "vehicle_dataset", "Annotations"
)

split_path = "/kaggle/working/vehicle_split_70_15_15.csv"

output_original = "/kaggle/working/yolo_original"
output_enhanced = "/kaggle/working/yolo_enhanced"

# ============================================================
# CLASS MAPPING
# ============================================================

class_map = {
    "Bicycle": 0,
    "Boat": 1,
    "Bus": 2,
    "Car": 3,
    "Motorbike": 4
}

# ============================================================
# LOAD LOCKED SPLIT
# ============================================================

split_df = pd.read_csv(split_path)

print("Loaded split rows:", len(split_df))

# ============================================================
# CREATE DIRECTORY STRUCTURE
# ============================================================

for dataset_root in [output_original, output_enhanced]:
    for split in ["train", "val", "test"]:
        os.makedirs(
            os.path.join(dataset_root, "images", split),
            exist_ok=True
        )
        os.makedirs(
            os.path.join(dataset_root, "labels", split),
            exist_ok=True
        )

# ============================================================
# CONVERSION FUNCTION
# ============================================================

def convert_annotation(annotation_path, img_w, img_h):

    yolo_lines = []

    with open(annotation_path, "r") as f:
        lines = f.readlines()

    for line in lines:

        line = line.strip()

        if not line or line.startswith("%"):
            continue

        parts = line.split()

        object_class = parts[0]

        if object_class not in class_map:
            continue

        x_left = float(parts[1])
        y_top = float(parts[2])
        box_w = float(parts[3])
        box_h = float(parts[4])

        # bbGt -> YOLO
        x_center = (x_left + box_w / 2) / img_w
        y_center = (y_top + box_h / 2) / img_h

        norm_w = box_w / img_w
        norm_h = box_h / img_h

        class_id = class_map[object_class]

        # Safety check
        if not (
            0 <= x_center <= 1 and
            0 <= y_center <= 1 and
            0 < norm_w <= 1 and
            0 < norm_h <= 1
        ):
            raise ValueError(
                f"Invalid YOLO box in {annotation_path}: "
                f"{object_class} "
                f"{x_center}, {y_center}, {norm_w}, {norm_h}"
            )

        yolo_lines.append(
            f"{class_id} "
            f"{x_center:.6f} "
            f"{y_center:.6f} "
            f"{norm_w:.6f} "
            f"{norm_h:.6f}"
        )

    return yolo_lines


# ============================================================
# BUILD DATASETS
# ============================================================

processed = 0
total_boxes = 0

for _, row in split_df.iterrows():

    class_name = row["class"]
    filename = row["filename"]
    split = row["split"]

    original_path = os.path.join(
        original_root,
        class_name,
        filename
    )

    enhanced_path = os.path.join(
        enhanced_root,
        class_name,
        filename
    )

    annotation_path = os.path.join(
        annotation_root,
        class_name,
        filename + ".txt"
    )

    # Read original only to get dimensions
    img = cv2.imread(original_path)

    if img is None:
        raise ValueError(
            f"Could not read image: {original_path}"
        )

    img_h, img_w = img.shape[:2]

    # Convert annotation
    yolo_lines = convert_annotation(
        annotation_path,
        img_w,
        img_h
    )

    if len(yolo_lines) == 0:
        raise ValueError(
            f"No valid vehicle boxes: {annotation_path}"
        )

    total_boxes += len(yolo_lines)

    # --------------------------------------------------------
    # IMPORTANT:
    # Prevent duplicate filenames from different class folders
    # --------------------------------------------------------

    safe_name = f"{class_name}_{filename}"

    label_name = os.path.splitext(safe_name)[0] + ".txt"

    # --------------------------------------------------------
    # ORIGINAL
    # --------------------------------------------------------

    shutil.copy2(
        original_path,
        os.path.join(
            output_original,
            "images",
            split,
            safe_name
        )
    )

    with open(
        os.path.join(
            output_original,
            "labels",
            split,
            label_name
        ),
        "w"
    ) as f:
        f.write("\n".join(yolo_lines))

    # --------------------------------------------------------
    # ENHANCED
    # --------------------------------------------------------

    shutil.copy2(
        enhanced_path,
        os.path.join(
            output_enhanced,
            "images",
            split,
            safe_name
        )
    )

    # SAME LABELS because geometry is unchanged
    with open(
        os.path.join(
            output_enhanced,
            "labels",
            split,
            label_name
        ),
        "w"
    ) as f:
        f.write("\n".join(yolo_lines))

    processed += 1

# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("FULL YOLO DATASET CREATION COMPLETE")
print("=" * 65)

print("\nImages processed:", processed)
print("Vehicle bounding boxes:", total_boxes)

for split in ["train", "val", "test"]:

    original_images = len(
        os.listdir(
            os.path.join(
                output_original,
                "images",
                split
            )
        )
    )

    original_labels = len(
        os.listdir(
            os.path.join(
                output_original,
                "labels",
                split
            )
        )
    )

    enhanced_images = len(
        os.listdir(
            os.path.join(
                output_enhanced,
                "images",
                split
            )
        )
    )

    enhanced_labels = len(
        os.listdir(
            os.path.join(
                output_enhanced,
                "labels",
                split
            )
        )
    )

    print(
        f"\n{split.upper()}:"
        f"\n  Original images : {original_images}"
        f"\n  Original labels : {original_labels}"
        f"\n  Enhanced images : {enhanced_images}"
        f"\n  Enhanced labels : {enhanced_labels}"
    )

print("\nExpected:")
print("Train = 2097")
print("Val   = 450")
print("Test  = 450")

print("\nOriginal dataset:", output_original)
print("Enhanced dataset:", output_enhanced)

print("=" * 65)

Loaded split rows: 2997

FULL YOLO DATASET CREATION COMPLETE

Images processed: 2997
Vehicle bounding boxes: 6680

TRAIN:
  Original images : 2097
  Original labels : 2097
  Enhanced images : 2097
  Enhanced labels : 2097

VAL:
  Original images : 450
  Original labels : 450
  Enhanced images : 450
  Enhanced labels : 450

TEST:
  Original images : 450
  Original labels : 450
  Enhanced images : 450
  Enhanced labels : 450

Expected:
Train = 2097
Val   = 450
Test  = 450

Original dataset: /kaggle/working/yolo_original
Enhanced dataset: /kaggle/working/yolo_enhanced


In [8]:
import os
from collections import Counter

# ============================================================
# PATHS
# ============================================================

datasets = {
    "Original": "/kaggle/working/yolo_original",
    "Enhanced": "/kaggle/working/yolo_enhanced"
}

class_names = {
    0: "Bicycle",
    1: "Boat",
    2: "Bus",
    3: "Car",
    4: "Motorbike"
}

# ============================================================
# AUDIT FUNCTION
# ============================================================

def audit_yolo_dataset(root):

    total_images = 0
    total_labels = 0
    total_boxes = 0

    class_counts = Counter()

    invalid_lines = []
    empty_labels = []
    missing_labels = []
    missing_images = []

    split_summary = {}

    for split in ["train", "val", "test"]:

        image_dir = os.path.join(root, "images", split)
        label_dir = os.path.join(root, "labels", split)

        images = sorted([
            f for f in os.listdir(image_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])

        labels = sorted([
            f for f in os.listdir(label_dir)
            if f.lower().endswith(".txt")
        ])

        total_images += len(images)
        total_labels += len(labels)

        split_boxes = 0

        # -------------------------------
        # Check images -> labels
        # -------------------------------

        for image_name in images:

            stem = os.path.splitext(image_name)[0]
            label_name = stem + ".txt"

            label_path = os.path.join(
                label_dir,
                label_name
            )

            if not os.path.exists(label_path):
                missing_labels.append(
                    f"{split}/{image_name}"
                )
                continue

            with open(label_path, "r") as f:
                lines = [
                    x.strip()
                    for x in f.readlines()
                    if x.strip()
                ]

            if len(lines) == 0:
                empty_labels.append(
                    f"{split}/{label_name}"
                )

            for line_no, line in enumerate(lines, start=1):

                parts = line.split()

                if len(parts) != 5:
                    invalid_lines.append(
                        (split, label_name, line_no, line)
                    )
                    continue

                try:
                    class_id = int(parts[0])

                    xc = float(parts[1])
                    yc = float(parts[2])
                    w = float(parts[3])
                    h = float(parts[4])

                except ValueError:

                    invalid_lines.append(
                        (split, label_name, line_no, line)
                    )
                    continue

                if (
                    class_id not in class_names
                    or not (0 <= xc <= 1)
                    or not (0 <= yc <= 1)
                    or not (0 < w <= 1)
                    or not (0 < h <= 1)
                ):
                    invalid_lines.append(
                        (split, label_name, line_no, line)
                    )
                    continue

                class_counts[class_id] += 1

                split_boxes += 1
                total_boxes += 1

        # -------------------------------
        # Check labels -> images
        # -------------------------------

        image_stems = {
            os.path.splitext(x)[0]
            for x in images
        }

        for label_name in labels:

            stem = os.path.splitext(label_name)[0]

            if stem not in image_stems:
                missing_images.append(
                    f"{split}/{label_name}"
                )

        split_summary[split] = {
            "images": len(images),
            "labels": len(labels),
            "boxes": split_boxes
        }

    return {
        "total_images": total_images,
        "total_labels": total_labels,
        "total_boxes": total_boxes,
        "class_counts": class_counts,
        "invalid_lines": invalid_lines,
        "empty_labels": empty_labels,
        "missing_labels": missing_labels,
        "missing_images": missing_images,
        "split_summary": split_summary
    }


# ============================================================
# RUN AUDIT
# ============================================================

results = {}

for name, root in datasets.items():

    results[name] = audit_yolo_dataset(root)

    r = results[name]

    print("\n" + "=" * 70)
    print(f"{name.upper()} YOLO DATASET AUDIT")
    print("=" * 70)

    for split in ["train", "val", "test"]:

        s = r["split_summary"][split]

        print(
            f"{split.upper():5} | "
            f"Images: {s['images']:4} | "
            f"Labels: {s['labels']:4} | "
            f"Boxes: {s['boxes']:4}"
        )

    print("\nTotal images :", r["total_images"])
    print("Total labels :", r["total_labels"])
    print("Total boxes  :", r["total_boxes"])

    print("\nBoxes by class:")

    for class_id, class_name in class_names.items():

        print(
            f"{class_id} {class_name:10}: "
            f"{r['class_counts'][class_id]}"
        )

    print("\nMissing labels :", len(r["missing_labels"]))
    print("Missing images :", len(r["missing_images"]))
    print("Empty labels   :", len(r["empty_labels"]))
    print("Invalid lines  :", len(r["invalid_lines"]))


# ============================================================
# ORIGINAL vs ENHANCED LABEL EQUALITY
# ============================================================

label_mismatches = []

for split in ["train", "val", "test"]:

    orig_dir = os.path.join(
        datasets["Original"],
        "labels",
        split
    )

    enh_dir = os.path.join(
        datasets["Enhanced"],
        "labels",
        split
    )

    for filename in os.listdir(orig_dir):

        orig_path = os.path.join(orig_dir, filename)
        enh_path = os.path.join(enh_dir, filename)

        with open(orig_path, "r") as f:
            orig_text = f.read()

        with open(enh_path, "r") as f:
            enh_text = f.read()

        if orig_text != enh_text:
            label_mismatches.append(
                f"{split}/{filename}"
            )

print("\n" + "=" * 70)
print("ORIGINAL vs ENHANCED LABEL CHECK")
print("=" * 70)

print("Label mismatches:", len(label_mismatches))

if (
    results["Original"]["total_images"] == 2997
    and results["Enhanced"]["total_images"] == 2997
    and len(results["Original"]["invalid_lines"]) == 0
    and len(results["Enhanced"]["invalid_lines"]) == 0
    and len(results["Original"]["missing_labels"]) == 0
    and len(results["Enhanced"]["missing_labels"]) == 0
    and len(label_mismatches) == 0
):
    print("✅ YOLO DATASET INTEGRITY PASSED")
else:
    print("❌ YOLO DATASET INTEGRITY ISSUE FOUND")

print("=" * 70)


ORIGINAL YOLO DATASET AUDIT
TRAIN | Images: 2097 | Labels: 2097 | Boxes: 4712
VAL   | Images:  450 | Labels:  450 | Boxes: 1002
TEST  | Images:  450 | Labels:  450 | Boxes:  966

Total images : 2997
Total labels : 2997
Total boxes  : 6680

Boxes by class:
0 Bicycle   : 1077
1 Boat      : 1377
2 Bus       : 689
3 Car       : 2501
4 Motorbike : 1036

Missing labels : 0
Missing images : 0
Empty labels   : 0
Invalid lines  : 0

ENHANCED YOLO DATASET AUDIT
TRAIN | Images: 2097 | Labels: 2097 | Boxes: 4712
VAL   | Images:  450 | Labels:  450 | Boxes: 1002
TEST  | Images:  450 | Labels:  450 | Boxes:  966

Total images : 2997
Total labels : 2997
Total boxes  : 6680

Boxes by class:
0 Bicycle   : 1077
1 Boat      : 1377
2 Bus       : 689
3 Car       : 2501
4 Motorbike : 1036

Missing labels : 0
Missing images : 0
Empty labels   : 0
Invalid lines  : 0

ORIGINAL vs ENHANCED LABEL CHECK
Label mismatches: 0
✅ YOLO DATASET INTEGRITY PASSED


In [9]:
import os
import yaml

# ============================================================
# DATASET PATHS
# ============================================================

original_root = "/kaggle/working/yolo_original"
enhanced_root = "/kaggle/working/yolo_enhanced"

# ============================================================
# CLASS NAMES
# ============================================================

class_names = {
    0: "Bicycle",
    1: "Boat",
    2: "Bus",
    3: "Car",
    4: "Motorbike"
}

# ============================================================
# ORIGINAL YAML
# ============================================================

original_yaml = {
    "path": original_root,
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": class_names
}

original_yaml_path = "/kaggle/working/original_vehicle.yaml"

with open(original_yaml_path, "w") as f:
    yaml.dump(
        original_yaml,
        f,
        sort_keys=False
    )

# ============================================================
# ENHANCED YAML
# ============================================================

enhanced_yaml = {
    "path": enhanced_root,
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": class_names
}

enhanced_yaml_path = "/kaggle/working/enhanced_vehicle.yaml"

with open(enhanced_yaml_path, "w") as f:
    yaml.dump(
        enhanced_yaml,
        f,
        sort_keys=False
    )

# ============================================================
# VERIFY
# ============================================================

print("=" * 65)
print("YOLO DATASET YAML CREATION")
print("=" * 65)

print("\nORIGINAL YAML:\n")
with open(original_yaml_path, "r") as f:
    print(f.read())

print("ENHANCED YAML:\n")
with open(enhanced_yaml_path, "r") as f:
    print(f.read())

print("Original YAML:", original_yaml_path)
print("Enhanced YAML:", enhanced_yaml_path)

print("\n✅ YOLO YAML FILES CREATED")
print("=" * 65)

YOLO DATASET YAML CREATION

ORIGINAL YAML:

path: /kaggle/working/yolo_original
train: images/train
val: images/val
test: images/test
names:
  0: Bicycle
  1: Boat
  2: Bus
  3: Car
  4: Motorbike

ENHANCED YAML:

path: /kaggle/working/yolo_enhanced
train: images/train
val: images/val
test: images/test
names:
  0: Bicycle
  1: Boat
  2: Bus
  3: Car
  4: Motorbike

Original YAML: /kaggle/working/original_vehicle.yaml
Enhanced YAML: /kaggle/working/enhanced_vehicle.yaml

✅ YOLO YAML FILES CREATED


# Yolo 11

In [10]:
# ============================================================
# CELL 58 — YOLO11 ENVIRONMENT SETUP
# ============================================================

!pip install -q ultralytics

from ultralytics import YOLO
import torch
import os
from pathlib import Path

print("=" * 70)
print("YOLO11 ENVIRONMENT CHECK")
print("=" * 70)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    device = 0
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / (1024**3),
            2
        ),
        "GB"
    )
else:
    device = "cpu"
    print("⚠️ CUDA GPU NOT AVAILABLE")

print("\nOriginal YAML exists:",
      os.path.exists("/kaggle/working/original_vehicle.yaml"))

print("Enhanced YAML exists:",
      os.path.exists("/kaggle/working/enhanced_vehicle.yaml"))

print("=" * 70)
print("✅ YOLO11 SETUP COMPLETE")
print("=" * 70)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 5.1 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
YOLO11 ENVIRONMENT CHECK
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB

Original YAML exists: True
Enhanced YAML exists: True
✅ YOLO11 SETUP COMPLETE


In [11]:
# ============================================================
# CELL 59 — YOLO11s ORIGINAL FINAL TRAINING
# ============================================================

from ultralytics import YOLO
import torch

print("=" * 70)
print("YOLO11s ORIGINAL — FINAL FINE-TUNING")
print("=" * 70)

device = 0 if torch.cuda.is_available() else "cpu"

# IMPORTANT:
# Fresh COCO-pretrained YOLO11s
# No checkpoint / no resume
model_original = YOLO("yolo11s.pt")

original_train_results = model_original.train(
    data="/kaggle/working/original_vehicle.yaml",

    epochs=100,
    patience=15,

    imgsz=640,
    batch=8,

    device=device,
    workers=2,

    seed=42,
    deterministic=True,

    pretrained=True,

    project="/kaggle/working/vehicle_detection_final",
    name="yolo11s_original_final",
    exist_ok=True,

    plots=True,
    verbose=True
)

ORIGINAL_BEST = (
    "/kaggle/working/vehicle_detection_final/"
    "yolo11s_original_final/weights/best.pt"
)

print("\n" + "=" * 70)
print("YOLO11s ORIGINAL TRAINING COMPLETE")
print("=" * 70)
print("Best model:", ORIGINAL_BEST)
print("Best model exists:", os.path.exists(ORIGINAL_BEST))
print("=" * 70)

YOLO11s ORIGINAL — FINAL FINE-TUNING
Ultralytics 8.4.133 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/original_vehicle.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scal

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


      2/100      3.29G      1.603      1.874      1.603          6        640: 100% ━━━━━━━━━━━━ 263/263 6.8it/s 38.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 8.8it/s 3.3s
                   all        450       1002      0.512      0.292      0.348      0.177

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/100      3.31G      1.672      2.052      1.651          6        640: 100% ━━━━━━━━━━━━ 263/263 6.8it/s 38.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 8.5it/s 3.4s
                   all        450       1002      0.537       0.41      0.445      0.237

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      4/100      3.31G      1.642          2      1.643          6        640: 100% ━━━━━━━━━━━━ 263/263 6.8it/s 38.4s
                 Class     Images  Instances      Box

In [12]:
# ============================================================
# CELL 60 — YOLO11s ENHANCED FINAL TRAINING
# ============================================================

from ultralytics import YOLO
import torch
import os

print("=" * 70)
print("YOLO11s ENHANCED — FINAL FINE-TUNING")
print("=" * 70)

device = 0 if torch.cuda.is_available() else "cpu"

# IMPORTANT:
# Independently start from fresh COCO-pretrained YOLO11s.
# DO NOT use Original-trained best.pt.
model_enhanced = YOLO("yolo11s.pt")

enhanced_train_results = model_enhanced.train(
    data="/kaggle/working/enhanced_vehicle.yaml",

    epochs=100,
    patience=15,

    imgsz=640,
    batch=8,

    device=device,
    workers=2,

    seed=42,
    deterministic=True,

    pretrained=True,

    project="/kaggle/working/vehicle_detection_final",
    name="yolo11s_enhanced_final",
    exist_ok=True,

    plots=True,
    verbose=True
)

ENHANCED_BEST = (
    "/kaggle/working/vehicle_detection_final/"
    "yolo11s_enhanced_final/weights/best.pt"
)

print("\n" + "=" * 70)
print("YOLO11s ENHANCED TRAINING COMPLETE")
print("=" * 70)
print("Best model:", ENHANCED_BEST)
print("Best model exists:", os.path.exists(ENHANCED_BEST))
print("=" * 70)

YOLO11s ENHANCED — FINAL FINE-TUNING
New https://pypi.org/project/ultralytics/8.4.134 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.133 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/enhanced_vehicle.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask

In [13]:
# ============================================================
# CELL 61 — TEST 1/4
# ORIGINAL-TRAINED → ORIGINAL TEST
# ============================================================

from ultralytics import YOLO
import torch
import json
import os

print("=" * 70)
print("YOLO11 TEST 1/4")
print("ORIGINAL-TRAINED → ORIGINAL TEST")
print("=" * 70)

device = 0 if torch.cuda.is_available() else "cpu"

model_path = (
    "/kaggle/working/vehicle_detection_final/"
    "yolo11s_original_final/weights/best.pt"
)

assert os.path.exists(model_path), f"Model not found: {model_path}"

model = YOLO(model_path)

metrics = model.val(
    data="/kaggle/working/original_vehicle.yaml",
    split="test",

    imgsz=640,
    batch=8,

    device=device,
    workers=2,

    project="/kaggle/working/vehicle_detection_test_yolo11",
    name="yolo11s_Otrain_Otest",
    exist_ok=True,

    plots=True,
    verbose=True
)

P = float(metrics.box.mp)
R = float(metrics.box.mr)
F1 = 2 * P * R / (P + R) if (P + R) > 0 else 0.0
MAP50 = float(metrics.box.map50)
MAP5095 = float(metrics.box.map)

result = {
    "condition": "Otrain_Otest",
    "precision": P,
    "recall": R,
    "f1": F1,
    "map50": MAP50,
    "map50_95": MAP5095
}

os.makedirs(
    "/kaggle/working/yolo11_final_results",
    exist_ok=True
)

with open(
    "/kaggle/working/yolo11_final_results/Otrain_Otest.json",
    "w"
) as f:
    json.dump(result, f, indent=4)

print("\n" + "=" * 70)
print("TEST 1 FINAL METRICS")
print("=" * 70)
print(f"Precision : {P*100:.2f}%")
print(f"Recall    : {R*100:.2f}%")
print(f"F1-score  : {F1*100:.2f}%")
print(f"mAP@50    : {MAP50*100:.2f}%")
print(f"mAP@50:95 : {MAP5095*100:.2f}%")
print("=" * 70)

YOLO11 TEST 1/4
ORIGINAL-TRAINED → ORIGINAL TEST
Ultralytics 8.4.133 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11s summary (fused): 101 layers, 9,414,735 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1988.7±904.2 MB/s, size: 125.9 KB)
val: Scanning /kaggle/working/yolo_original/labels/test... 450 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 450/450 1.0Kit/s 0.4s
val: /kaggle/working/yolo_original/images/test/Car_2015_02634.jpg: corrupt JPEG restored and saved
val: New cache created: /kaggle/working/yolo_original/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 57/57 10.7it/s 5.3s
                   all        450        966      0.861      0.756      0.843      0.567
               Bicycle        108        172      0.853      0.791      0.847       0.56
                  Boat        102        209      0.808      0.623      0.714      0

In [14]:
# ============================================================
# CELL 62 — TEST 2/4
# ORIGINAL-TRAINED → ENHANCED TEST
# ============================================================

from ultralytics import YOLO
import torch
import json
import os

print("=" * 70)
print("YOLO11 TEST 2/4")
print("ORIGINAL-TRAINED → ENHANCED TEST")
print("=" * 70)

device = 0 if torch.cuda.is_available() else "cpu"

model_path = (
    "/kaggle/working/vehicle_detection_final/"
    "yolo11s_original_final/weights/best.pt"
)

assert os.path.exists(model_path), f"Model not found: {model_path}"

model = YOLO(model_path)

metrics = model.val(
    data="/kaggle/working/enhanced_vehicle.yaml",
    split="test",

    imgsz=640,
    batch=8,

    device=device,
    workers=2,

    project="/kaggle/working/vehicle_detection_test_yolo11",
    name="yolo11s_Otrain_Etest",
    exist_ok=True,

    plots=True,
    verbose=True
)

P = float(metrics.box.mp)
R = float(metrics.box.mr)
F1 = 2 * P * R / (P + R) if (P + R) > 0 else 0.0
MAP50 = float(metrics.box.map50)
MAP5095 = float(metrics.box.map)

result = {
    "condition": "Otrain_Etest",
    "precision": P,
    "recall": R,
    "f1": F1,
    "map50": MAP50,
    "map50_95": MAP5095
}

with open(
    "/kaggle/working/yolo11_final_results/Otrain_Etest.json",
    "w"
) as f:
    json.dump(result, f, indent=4)

print("\n" + "=" * 70)
print("TEST 2 FINAL METRICS")
print("=" * 70)
print(f"Precision : {P*100:.2f}%")
print(f"Recall    : {R*100:.2f}%")
print(f"F1-score  : {F1*100:.2f}%")
print(f"mAP@50    : {MAP50*100:.2f}%")
print(f"mAP@50:95 : {MAP5095*100:.2f}%")
print("=" * 70)

YOLO11 TEST 2/4
ORIGINAL-TRAINED → ENHANCED TEST
Ultralytics 8.4.133 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11s summary (fused): 101 layers, 9,414,735 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2220.9±872.1 MB/s, size: 429.9 KB)
val: Scanning /kaggle/working/yolo_enhanced/labels/test... 450 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 450/450 902.1it/s 0.5s
val: New cache created: /kaggle/working/yolo_enhanced/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 57/57 10.6it/s 5.4s
                   all        450        966      0.838      0.682      0.789      0.524
               Bicycle        108        172      0.818       0.73      0.793      0.538
                  Boat        102        209      0.798      0.598      0.691      0.364
                   Bus         86         93      0.925      0.839      0.918      0.733
    

In [15]:
# ============================================================
# CELL 63 — TEST 3/4
# ENHANCED-TRAINED → ORIGINAL TEST
# ============================================================

from ultralytics import YOLO
import torch
import json
import os

print("=" * 70)
print("YOLO11 TEST 3/4")
print("ENHANCED-TRAINED → ORIGINAL TEST")
print("=" * 70)

device = 0 if torch.cuda.is_available() else "cpu"

model_path = (
    "/kaggle/working/vehicle_detection_final/"
    "yolo11s_enhanced_final/weights/best.pt"
)

assert os.path.exists(model_path), f"Model not found: {model_path}"

model = YOLO(model_path)

metrics = model.val(
    data="/kaggle/working/original_vehicle.yaml",
    split="test",

    imgsz=640,
    batch=8,

    device=device,
    workers=2,

    project="/kaggle/working/vehicle_detection_test_yolo11",
    name="yolo11s_Etrain_Otest",
    exist_ok=True,

    plots=True,
    verbose=True
)

P = float(metrics.box.mp)
R = float(metrics.box.mr)
F1 = 2 * P * R / (P + R) if (P + R) > 0 else 0.0
MAP50 = float(metrics.box.map50)
MAP5095 = float(metrics.box.map)

result = {
    "condition": "Etrain_Otest",
    "precision": P,
    "recall": R,
    "f1": F1,
    "map50": MAP50,
    "map50_95": MAP5095
}

with open(
    "/kaggle/working/yolo11_final_results/Etrain_Otest.json",
    "w"
) as f:
    json.dump(result, f, indent=4)

print("\n" + "=" * 70)
print("TEST 3 FINAL METRICS")
print("=" * 70)
print(f"Precision : {P*100:.2f}%")
print(f"Recall    : {R*100:.2f}%")
print(f"F1-score  : {F1*100:.2f}%")
print(f"mAP@50    : {MAP50*100:.2f}%")
print(f"mAP@50:95 : {MAP5095*100:.2f}%")
print("=" * 70)

YOLO11 TEST 3/4
ENHANCED-TRAINED → ORIGINAL TEST
Ultralytics 8.4.133 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11s summary (fused): 101 layers, 9,414,735 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1877.9±392.5 MB/s, size: 126.3 KB)
val: Scanning /kaggle/working/yolo_original/labels/test.cache... 450 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 450/450 157.3Mit/s 0.0s
val: /kaggle/working/yolo_original/images/test/Car_2015_02634.jpg: corrupt JPEG restored and saved
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 57/57 10.7it/s 5.3s
                   all        450        966      0.873      0.685      0.793      0.533
               Bicycle        108        172       0.88       0.68      0.805      0.532
                  Boat        102        209      0.835      0.512      0.652      0.365
                   Bus         86         93      0.918    

In [16]:
# ============================================================
# CELL 64 — TEST 4/4
# ENHANCED-TRAINED → ENHANCED TEST
# ============================================================

from ultralytics import YOLO
import torch
import json
import os

print("=" * 70)
print("YOLO11 TEST 4/4")
print("ENHANCED-TRAINED → ENHANCED TEST")
print("=" * 70)

device = 0 if torch.cuda.is_available() else "cpu"

model_path = (
    "/kaggle/working/vehicle_detection_final/"
    "yolo11s_enhanced_final/weights/best.pt"
)

assert os.path.exists(model_path), f"Model not found: {model_path}"

model = YOLO(model_path)

metrics = model.val(
    data="/kaggle/working/enhanced_vehicle.yaml",
    split="test",

    imgsz=640,
    batch=8,

    device=device,
    workers=2,

    project="/kaggle/working/vehicle_detection_test_yolo11",
    name="yolo11s_Etrain_Etest",
    exist_ok=True,

    plots=True,
    verbose=True
)

P = float(metrics.box.mp)
R = float(metrics.box.mr)
F1 = 2 * P * R / (P + R) if (P + R) > 0 else 0.0
MAP50 = float(metrics.box.map50)
MAP5095 = float(metrics.box.map)

result = {
    "condition": "Etrain_Etest",
    "precision": P,
    "recall": R,
    "f1": F1,
    "map50": MAP50,
    "map50_95": MAP5095
}

with open(
    "/kaggle/working/yolo11_final_results/Etrain_Etest.json",
    "w"
) as f:
    json.dump(result, f, indent=4)

print("\n" + "=" * 70)
print("TEST 4 FINAL METRICS")
print("=" * 70)
print(f"Precision : {P*100:.2f}%")
print(f"Recall    : {R*100:.2f}%")
print(f"F1-score  : {F1*100:.2f}%")
print(f"mAP@50    : {MAP50*100:.2f}%")
print(f"mAP@50:95 : {MAP5095*100:.2f}%")
print("=" * 70)

YOLO11 TEST 4/4
ENHANCED-TRAINED → ENHANCED TEST
Ultralytics 8.4.133 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11s summary (fused): 101 layers, 9,414,735 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2002.6±722.4 MB/s, size: 105.5 KB)
val: Scanning /kaggle/working/yolo_enhanced/labels/test.cache... 450 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 450/450 209.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 57/57 10.6it/s 5.4s
                   all        450        966      0.814       0.75      0.804      0.539
               Bicycle        108        172      0.809      0.744        0.8      0.534
                  Boat        102        209      0.748      0.651      0.698      0.373
                   Bus         86         93       0.93      0.892      0.935       0.77
                   Car        154        346      0.827      0.731   

In [17]:
# ============================================================
# CELL 65 — YOLO11 CONTROLLED FPS BENCHMARK
# ALL FOUR 2×2 CONDITIONS
# ============================================================

from ultralytics import YOLO
from pathlib import Path
import torch
import time
import numpy as np
import pandas as pd
import os

print("=" * 75)
print("YOLO11 CONTROLLED SINGLE-IMAGE DETECTOR FPS BENCHMARK")
print("=" * 75)

device = 0 if torch.cuda.is_available() else "cpu"

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is required for the controlled GPU FPS benchmark."
    )

# ============================================================
# MODEL PATHS
# ============================================================

ORIGINAL_MODEL_PATH = (
    "/kaggle/working/vehicle_detection_final/"
    "yolo11s_original_final/weights/best.pt"
)

ENHANCED_MODEL_PATH = (
    "/kaggle/working/vehicle_detection_final/"
    "yolo11s_enhanced_final/weights/best.pt"
)

assert os.path.exists(ORIGINAL_MODEL_PATH)
assert os.path.exists(ENHANCED_MODEL_PATH)

# ============================================================
# TEST IMAGE DIRECTORIES
# ============================================================

ORIGINAL_TEST_DIR = Path(
    "/kaggle/working/yolo_original/images/test"
)

ENHANCED_TEST_DIR = Path(
    "/kaggle/working/yolo_enhanced/images/test"
)

VALID_SUFFIXES = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
    ".webp"
}

# ============================================================
# IMAGE COLLECTION
# ============================================================

def get_test_images(folder):

    images = sorted([
        p for p in folder.iterdir()
        if p.is_file()
        and p.suffix.lower() in VALID_SUFFIXES
    ])

    print(f"{folder}: {len(images)} images")

    if len(images) != 450:
        raise ValueError(
            f"Expected 450 test images, found {len(images)} in {folder}"
        )

    return images


original_test_images = get_test_images(
    ORIGINAL_TEST_DIR
)

enhanced_test_images = get_test_images(
    ENHANCED_TEST_DIR
)

# ============================================================
# BENCHMARK FUNCTION
# ============================================================

def benchmark_model(
    model_path,
    image_paths,
    condition,
    warmup_count=20
):

    print("\n" + "=" * 75)
    print("BENCHMARK:", condition)
    print("=" * 75)

    model = YOLO(model_path)

    # --------------------------------------------------------
    # WARM-UP
    # --------------------------------------------------------

    print(f"Warm-up: {warmup_count} images")

    for img_path in image_paths[:warmup_count]:

        model.predict(
            source=str(img_path),
            imgsz=640,
            device=device,
            verbose=False,
            save=False
        )

    torch.cuda.synchronize()

    # --------------------------------------------------------
    # TIMED SINGLE-IMAGE INFERENCE
    # --------------------------------------------------------

    latencies_ms = []

    for i, img_path in enumerate(image_paths, start=1):

        torch.cuda.synchronize()

        start = time.perf_counter()

        model.predict(
            source=str(img_path),
            imgsz=640,
            device=device,
            verbose=False,
            save=False
        )

        torch.cuda.synchronize()

        end = time.perf_counter()

        latency_ms = (end - start) * 1000.0

        latencies_ms.append(latency_ms)

    latencies_ms = np.array(
        latencies_ms,
        dtype=np.float64
    )

    # --------------------------------------------------------
    # SUMMARY
    # --------------------------------------------------------

    total_seconds = latencies_ms.sum() / 1000.0

    overall_fps = (
        len(image_paths) / total_seconds
    )

    mean_latency = latencies_ms.mean()
    median_latency = np.median(latencies_ms)

    median_latency_fps = (
        1000.0 / median_latency
    )

    minimum_latency = latencies_ms.min()
    maximum_latency = latencies_ms.max()

    print("\nImages:", len(image_paths))
    print(f"Total timed seconds : {total_seconds:.3f}")
    print(f"Mean latency        : {mean_latency:.3f} ms/image")
    print(f"Median latency      : {median_latency:.3f} ms/image")
    print(f"Overall FPS         : {overall_fps:.2f}")
    print(f"Median-latency FPS  : {median_latency_fps:.2f}")
    print(f"Minimum latency     : {minimum_latency:.3f} ms")
    print(f"Maximum latency     : {maximum_latency:.3f} ms")

    # Release current model before next condition
    del model
    torch.cuda.empty_cache()

    return {
        "Condition": condition,
        "Images": len(image_paths),
        "Total_Time_s": total_seconds,
        "Mean_Latency_ms": mean_latency,
        "Median_Latency_ms": median_latency,
        "FPS": overall_fps,
        "Median_Latency_FPS": median_latency_fps,
        "Min_Latency_ms": minimum_latency,
        "Max_Latency_ms": maximum_latency
    }


# ============================================================
# RUN 2×2 BENCHMARK
# ============================================================

fps_results = []

# 1. Original-trained -> Original test
fps_results.append(
    benchmark_model(
        ORIGINAL_MODEL_PATH,
        original_test_images,
        "Otrain_Otest"
    )
)

# 2. Original-trained -> Enhanced test
fps_results.append(
    benchmark_model(
        ORIGINAL_MODEL_PATH,
        enhanced_test_images,
        "Otrain_Etest"
    )
)

# 3. Enhanced-trained -> Original test
fps_results.append(
    benchmark_model(
        ENHANCED_MODEL_PATH,
        original_test_images,
        "Etrain_Otest"
    )
)

# 4. Enhanced-trained -> Enhanced test
fps_results.append(
    benchmark_model(
        ENHANCED_MODEL_PATH,
        enhanced_test_images,
        "Etrain_Etest"
    )
)

# ============================================================
# SAVE FPS RESULTS
# ============================================================

fps_df = pd.DataFrame(fps_results)

fps_csv = (
    "/kaggle/working/yolo11_final_results/"
    "yolo11_controlled_fps.csv"
)

fps_df.to_csv(
    fps_csv,
    index=False
)

print("\n" + "=" * 75)
print("FINAL YOLO11 CONTROLLED FPS RESULTS")
print("=" * 75)

print(
    fps_df[
        [
            "Condition",
            "Images",
            "Mean_Latency_ms",
            "Median_Latency_ms",
            "FPS"
        ]
    ].to_string(index=False)
)

print("\nSaved:")
print(fps_csv)

print("\nNOTE:")
print(
    "FPS measures detector processing on already-created images."
)
print(
    "RetinexFormer enhancement preprocessing time is NOT included."
)

print("=" * 75)

YOLO11 CONTROLLED SINGLE-IMAGE DETECTOR FPS BENCHMARK
/kaggle/working/yolo_original/images/test: 450 images
/kaggle/working/yolo_enhanced/images/test: 450 images

BENCHMARK: Otrain_Otest
Warm-up: 20 images

Images: 450
Total timed seconds : 8.380
Mean latency        : 18.623 ms/image
Median latency      : 13.335 ms/image
Overall FPS         : 53.70
Median-latency FPS  : 74.99
Minimum latency     : 10.446 ms
Maximum latency     : 111.441 ms

BENCHMARK: Otrain_Etest
Warm-up: 20 images

Images: 450
Total timed seconds : 7.687
Mean latency        : 17.082 ms/image
Median latency      : 13.199 ms/image
Overall FPS         : 58.54
Median-latency FPS  : 75.76
Minimum latency     : 10.451 ms
Maximum latency     : 135.000 ms

BENCHMARK: Etrain_Otest
Warm-up: 20 images

Images: 450
Total timed seconds : 7.672
Mean latency        : 17.049 ms/image
Median latency      : 13.334 ms/image
Overall FPS         : 58.66
Median-latency FPS  : 75.00
Minimum latency     : 10.550 ms
Maximum latency     : 114

In [18]:
# ============================================================
# CELL 66 — FINAL YOLO11 2×2 RESULTS TABLE
# ============================================================

import json
import pandas as pd
from pathlib import Path

RESULT_DIR = Path(
    "/kaggle/working/yolo11_final_results"
)

conditions = [
    "Otrain_Otest",
    "Otrain_Etest",
    "Etrain_Otest",
    "Etrain_Etest"
]

rows = []

for condition in conditions:

    json_path = RESULT_DIR / f"{condition}.json"

    if not json_path.exists():
        raise FileNotFoundError(
            f"Missing test result: {json_path}"
        )

    with open(json_path, "r") as f:
        r = json.load(f)

    rows.append(r)

metrics_df = pd.DataFrame(rows)

fps_path = (
    RESULT_DIR /
    "yolo11_controlled_fps.csv"
)

fps_df = pd.read_csv(fps_path)

final_df = metrics_df.merge(
    fps_df[
        [
            "Condition",
            "FPS",
            "Mean_Latency_ms",
            "Median_Latency_ms"
        ]
    ],
    left_on="condition",
    right_on="Condition",
    how="left"
)

final_df["Precision_%"] = (
    final_df["precision"] * 100
)

final_df["Recall_%"] = (
    final_df["recall"] * 100
)

final_df["F1_%"] = (
    final_df["f1"] * 100
)

final_df["mAP50_%"] = (
    final_df["map50"] * 100
)

final_df["mAP50_95_%"] = (
    final_df["map50_95"] * 100
)

final_table = final_df[
    [
        "condition",
        "Precision_%",
        "Recall_%",
        "F1_%",
        "mAP50_%",
        "mAP50_95_%",
        "FPS"
    ]
].copy()

final_table.columns = [
    "Train→Test",
    "Precision (%)",
    "Recall (%)",
    "F1 (%)",
    "mAP@50 (%)",
    "mAP@50:95 (%)",
    "FPS"
]

for col in final_table.columns[1:]:
    final_table[col] = final_table[col].round(2)

final_csv = (
    RESULT_DIR /
    "YOLO11_FINAL_2x2_RESULTS.csv"
)

final_table.to_csv(
    final_csv,
    index=False
)

print("=" * 85)
print("YOLO11s FINAL 2×2 TEST RESULTS")
print("=" * 85)

print(
    final_table.to_string(
        index=False
    )
)

print("\nSaved:")
print(final_csv)

print("=" * 85)

YOLO11s FINAL 2×2 TEST RESULTS
  Train→Test  Precision (%)  Recall (%)  F1 (%)  mAP@50 (%)  mAP@50:95 (%)   FPS
Otrain_Otest          86.08       75.63   80.52       84.33          56.72 53.70
Otrain_Etest          83.79       68.18   75.19       78.92          52.44 58.54
Etrain_Otest          87.27       68.46   76.73       79.32          53.28 58.66
Etrain_Etest          81.39       75.03   78.08       80.41          53.94 59.24

Saved:
/kaggle/working/yolo11_final_results/YOLO11_FINAL_2x2_RESULTS.csv
